# Analisis sensitivitas audit kesetaraan (untuk menjawab reviewer)
Tiga pertanyaan:
1. **Ambang rujukan lain** (0,10 dan 0,20 selain 0,155): apakah kesenjangan siapa yang terlewat tetap ada?
2. **Definisi hs-CRP ≥2 mg/L** (penanda risiko ACC/AHA) selain ≥3 mg/L.
3. **Bobot survei** (`pwt14dbsxa`): apakah FNR per kelompok berubah bila ditimbang?

Prosedur sama dengan `audit_kesetaraan_hsys.ipynb`: CV 5-fold × 3 per rumah tangga + Platt di dalam fold.
Dijalankan **di komputer sendiri**; yang dibagikan ke Claude hanya tabel ringkasan (xlsx). Simpan notebook (Ctrl+S).
Waktu: ±10–15 menit.

In [1]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = r"F:\SINTA 3\crp-ifls5-ml"
OUT = os.path.join(ROOT, "results")
DATA_FILE = os.path.join(ROOT, "data", "crp_ifls5_model.csv")
SEED, REP, NB = 42, 3, 300
FIVE = ["bmi", "female", "age", "srh_poor", "urban"]
rng = np.random.default_rng(SEED)

df = pd.read_csv(DATA_FILE, dtype={"pidlink": str, "hhid14": str})
g = df.hhid14.values
X = df[FIVE].astype(float)
prov = pd.to_numeric(df.province, errors="coerce")
REGION = {1: "Sumatra", 3: "Java", 5: "Bali–Nusa Tenggara", 6: "Kalimantan", 7: "Sulawesi"}
q = pd.qcut(df.lnpce.rank(method="first"), 5, labels=False) + 1
GROUPS = {
    "Sex": df.female.map({0: "Male", 1: "Female"}),
    "Age group": pd.cut(df.age, [0, 39, 59, 200], labels=["15–39", "40–59", "≥60"]).astype(str),
    "Residence": df.urban.map({0: "Rural", 1: "Urban"}),
    "Expenditure": q.map({1: "Q1–Q2 (poorest)", 2: "Q1–Q2 (poorest)", 3: "Q3", 4: "Q4–Q5 (richest)", 5: "Q4–Q5 (richest)"}),
    "Region": (prov // 10).map(REGION).fillna("Other"),
}
GROUPS = {k: v.astype(str).values for k, v in GROUPS.items()}
PAIRS = {"Sex": ("Male", "Female"), "Residence": ("Rural", "Urban"),
         "Expenditure": ("Q1–Q2 (poorest)", "Q4–Q5 (richest)"), "Age group": ("15–39", "40–59")}

HAS_CRP = "crp_plas_equi" in df.columns
HAS_W = "pwt14dbsxa" in df.columns
print("n =", len(df), "| kolom hs-CRP kontinu:", HAS_CRP, "| kolom bobot survei:", HAS_W)

n = 5980 | kolom hs-CRP kontinu: True | kolom bobot survei: True


## 1. Fungsi bersama (prediksi OOF terkalibrasi, FNR per kelompok, CI bootstrap)

In [2]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def model():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("m", LogisticRegression(class_weight="balanced", max_iter=2000))])
def platt(p, yt):
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p).reshape(-1, 1), yt)
    return lambda s: lr.predict_proba(logit(s).reshape(-1, 1))[:, 1]

def oof(y):
    P = np.zeros((REP, len(y)))
    for r in range(REP):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(X, y, g):
            m = model().fit(X.iloc[tr], y[tr]); pin = np.zeros(len(tr))
            for a, b in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(X.iloc[tr], y[tr], g[tr]):
                pin[b] = model().fit(X.iloc[tr].iloc[a], y[tr][a]).predict_proba(X.iloc[tr].iloc[b])[:, 1]
            P[r, te] = platt(pin, y[tr])(m.predict_proba(X.iloc[te])[:, 1])
    return P.mean(0)

def fnr(y, p, t, m, w=None):
    c = m & (y == 1)
    w = np.ones(len(y)) if w is None else w
    return (w[c] * (p[c] < t)).sum() / max(w[c].sum(), 1e-9) * 100

HH = pd.Series(g); uniq = HH.unique(); ix = HH.groupby(HH).indices
BOOT = [np.concatenate([ix[h] for h in rng.choice(uniq, len(uniq))]) for _ in range(NB)]

def audit(y, p, t, label, w=None):
    """Satu baris per kelompok + baris selisih FNR (dengan CI bootstrap) untuk pasangan utama."""
    rows = []
    allm = np.ones(len(y), bool)
    flag = p >= t
    rows.append({"Scenario": label, "Attribute": "All", "Group": "All adults",
                 "Cases": int(y.sum()), "Flagged (%)": flag.mean() * 100,
                 "FNR (%)": fnr(y, p, t, allm, w), "95% CI": ""})
    for attr, lab in GROUPS.items():
        for grp in [x for x in pd.unique(lab) if x not in ("Other", "nan")]:
            m = lab == grp
            bs = [fnr(y[b], p[b], t, m[b], None if w is None else w[b]) for b in BOOT]
            rows.append({"Scenario": label, "Attribute": attr, "Group": grp, "Cases": int(y[m].sum()),
                         "Flagged (%)": flag[m].mean() * 100, "FNR (%)": fnr(y, p, t, m, w),
                         "95% CI": f"{np.percentile(bs, 2.5):.1f}–{np.percentile(bs, 97.5):.1f}"})
    gaps = []
    for attr, (a, r_) in PAIRS.items():
        lab = GROUPS[attr]
        def gap(b):
            wb = None if w is None else w[b]
            return fnr(y[b], p[b], t, lab[b] == a, wb) - fnr(y[b], p[b], t, lab[b] == r_, wb)
        bs = np.array([gap(b) for b in BOOT])
        gaps.append({"Scenario": label, "Comparison": f"{a} vs {r_}", "FNR gap (pp)": gap(np.arange(len(y))),
                     "95% CI": f"{np.percentile(bs, 2.5):.1f} to {np.percentile(bs, 97.5):.1f}",
                     "p (bootstrap)": 2 * min((bs <= 0).mean(), (bs >= 0).mean())})
    regs = [fnr(y, p, t, GROUPS["Region"] == rg, w) for rg in REGION.values()]
    summ = {"Scenario": label, "Threshold": t, "Prevalence (%)": y.mean() * 100,
            "AUC": roc_auc_score(y, p, sample_weight=w), "Flagged (%)": flag.mean() * 100,
            "Sensitivity": 1 - fnr(y, p, t, allm, w) / 100,
            "FNR men − women (pp)": gaps[0]["FNR gap (pp)"], "FNR rural − urban (pp)": gaps[1]["FNR gap (pp)"],
            "FNR poorest − richest (pp)": gaps[2]["FNR gap (pp)"], "Range across regions (pp)": max(regs) - min(regs)}
    return rows, gaps, summ

## 2. Prediksi dasar (hs-CRP ≥3 mg/L) — harus sama dengan naskah (AUC 0,677)

In [3]:
t0 = time.time()
y3 = df.crp_high.values.astype(int)
p3 = oof(y3)
print(f"hs-CRP ≥3: AUC {roc_auc_score(y3, p3):.3f} | O/E {y3.mean()/p3.mean():.3f} | {(time.time()-t0)/60:.1f} menit")

hs-CRP ≥3: AUC 0.677 | O/E 1.001 | 0.4 menit


## 3. Analisis A — ambang rujukan 0,10 / 0,155 / 0,20

In [4]:
ROWS, GAPS, SUMM = [], [], []
for t in [0.10, 0.155, 0.20]:
    r, gp, s = audit(y3, p3, t, f"A. hs-CRP ≥3, threshold {t}")
    ROWS += r; GAPS += gp; SUMM.append(s)
print(pd.DataFrame(SUMM).round(3).to_string(index=False))

                     Scenario  Threshold  Prevalence (%)   AUC  Flagged (%)  Sensitivity  FNR men − women (pp)  FNR rural − urban (pp)  FNR poorest − richest (pp)  Range across regions (pp)
  A. hs-CRP ≥3, threshold 0.1      0.100          15.117 0.677       66.388        0.819                24.903                  14.656                      10.923                     13.752
A. hs-CRP ≥3, threshold 0.155      0.155          15.117 0.677       36.756        0.595                37.511                  24.336                      17.337                     20.560
  A. hs-CRP ≥3, threshold 0.2      0.200          15.117 0.677       22.224        0.431                37.800                  25.113                      19.269                     17.272


## 4. Analisis B — definisi hs-CRP ≥2 mg/L
Model dilatih ulang untuk luaran baru. Ambang dipilih dengan prinsip yang sama seperti 0,155 (≈ prevalensi nasional),
dan juga pada ambang yang menandai proporsi orang yang sama dengan aturan utama (36,8%).

In [5]:
if HAS_CRP:
    y2 = (pd.to_numeric(df.crp_plas_equi, errors="coerce") >= 2).astype(int).values
    t1 = time.time(); p2 = oof(y2)
    print(f"hs-CRP ≥2: prevalensi {y2.mean()*100:.1f}% | AUC {roc_auc_score(y2, p2):.3f} | {(time.time()-t1)/60:.1f} menit")
    t_prev = round(y2.mean(), 3)
    t_same = float(np.quantile(p2, 1 - (p3 >= 0.155).mean()))
    for t, nm in [(t_prev, "≈ prevalence"), (t_same, "same share flagged as main rule")]:
        r, gp, s = audit(y2, p2, t, f"B. hs-CRP ≥2, threshold {t:.3f} ({nm})")
        ROWS += r; GAPS += gp; SUMM.append(s)
    print(pd.DataFrame(SUMM[-2:]).round(3).to_string(index=False))
else:
    print("Kolom crp_plas_equi tidak ada: analisis B dilewati.")

hs-CRP ≥2: prevalensi 23.6% | AUC 0.681 | 0.4 menit
                                                       Scenario  Threshold  Prevalence (%)   AUC  Flagged (%)  Sensitivity  FNR men − women (pp)  FNR rural − urban (pp)  FNR poorest − richest (pp)  Range across regions (pp)
                   B. hs-CRP ≥2, threshold 0.236 (≈ prevalence)      0.236          23.629 0.681       40.736        0.614                34.274                  27.750                      14.044                     10.726
B. hs-CRP ≥2, threshold 0.250 (same share flagged as main rule)      0.250          23.629 0.681       36.756        0.580                33.547                  28.741                      13.229                      8.880


## 5. Analisis C — FNR per kelompok dengan bobot survei (hs-CRP ≥3, ambang 0,155)
CI bootstrap tetap per rumah tangga; bobot ikut dalam penghitungan FNR.

In [6]:
if HAS_W:
    w = pd.to_numeric(df.pwt14dbsxa, errors="coerce")
    print("Bobot hilang:", int(w.isna().sum()), "→ diisi median")
    w = w.fillna(w.median()).values
    r, gp, s = audit(y3, p3, 0.155, "C. hs-CRP ≥3, threshold 0.155, survey-weighted", w)
    ROWS += r; GAPS += gp; SUMM.append(s)
    print(pd.DataFrame(SUMM[-1:]).round(3).to_string(index=False))
else:
    print("Kolom pwt14dbsxa tidak ada: analisis C dilewati.")

Bobot hilang: 78 → diisi median
                                      Scenario  Threshold  Prevalence (%)   AUC  Flagged (%)  Sensitivity  FNR men − women (pp)  FNR rural − urban (pp)  FNR poorest − richest (pp)  Range across regions (pp)
C. hs-CRP ≥3, threshold 0.155, survey-weighted      0.155          15.117 0.687       36.756        0.578                38.864                  24.139                      16.232                     22.039


## 6. Simpan ringkasan

In [7]:
S = pd.DataFrame(SUMM); G = pd.DataFrame(GAPS); R = pd.DataFrame(ROWS)
pd.set_option("display.width", 250)
print(S.round(3).to_string(index=False)); print(); print(G.round(3).to_string(index=False))
with pd.ExcelWriter(os.path.join(OUT, "sensitivitas_hsys.xlsx")) as wr:
    S.round(4).to_excel(wr, sheet_name="ringkasan", index=False)
    G.round(4).to_excel(wr, sheet_name="selisih_FNR", index=False)
    R.round(4).to_excel(wr, sheet_name="per_kelompok", index=False)
print("Tersimpan:", os.path.join(OUT, "sensitivitas_hsys.xlsx"), "| Simpan notebook (Ctrl+S) lalu kabari Claude.")

                                                       Scenario  Threshold  Prevalence (%)   AUC  Flagged (%)  Sensitivity  FNR men − women (pp)  FNR rural − urban (pp)  FNR poorest − richest (pp)  Range across regions (pp)
                                    A. hs-CRP ≥3, threshold 0.1      0.100          15.117 0.677       66.388        0.819                24.903                  14.656                      10.923                     13.752
                                  A. hs-CRP ≥3, threshold 0.155      0.155          15.117 0.677       36.756        0.595                37.511                  24.336                      17.337                     20.560
                                    A. hs-CRP ≥3, threshold 0.2      0.200          15.117 0.677       22.224        0.431                37.800                  25.113                      19.269                     17.272
                   B. hs-CRP ≥2, threshold 0.236 (≈ prevalence)      0.236          23.629 0.681       4